Load schema definitions and config

In [0]:
%run ../config/config

In [0]:
dbutils.widgets.text("batch_id","")
batch_id=dbutils.widgets.get("batch_id")

In [0]:
silver_table = f"{catalog}.{silver_schema}.complete_data"
gold_table = f"{catalog}.{gold_schema}.dim_aircrafts"

Read silver Delta table into a DataFrame

In [0]:
from pyspark.sql import functions as F

silver_complete_df=(
    spark.read
    .format("delta")
    .table(silver_table)
    .filter(F.col("batch_id")==batch_id)
)

Select and rename columns for clarity and unification to create dimension table

In [0]:
gold_dim_aircrafts_df=(
    silver_complete_df
    .select(
        "tail_number_key",
        "aircraft_year_of_manufacture",
        "aircraft_manufacturer",
        "aircraft_type",
        "aircraft_range",
        "aircraft_width"
    )
)

Keep distinct airplane records for aircraft lookup table

In [0]:
gold_dim_aircrafts_df = gold_dim_aircrafts_df.distinct()

Write DataFrame to gold dim_aircrafts Delta table

In [0]:
if not spark.catalog.tableExists(gold_table):

    gold_dim_aircrafts_df_write = (
        gold_dim_aircrafts_df
        .write
        .format("delta")
        .mode("overwrite")
        .saveAsTable(gold_table)
    )

else:

    from delta.tables import DeltaTable

    delta_table = DeltaTable.forName(spark, gold_table)
    (
        delta_table.alias("t")
        .merge(
            gold_dim_aircrafts_df.alias("s"),
            "t.tail_number_key = s.tail_number_key"
        )
        .whenMatchedUpdate(
            set={
                "tail_number_key": "s.tail_number_key",
                "aircraft_year_of_manufacture": "s.aircraft_year_of_manufacture",
                "aircraft_manufacturer": "s.aircraft_manufacturer",
                "aircraft_type": "s.aircraft_type",
                "aircraft_range": "s.aircraft_range",
                "aircraft_width": "s.aircraft_width"
            }
        )
        .whenNotMatchedInsertAll()
        .execute()
    ) 